# Open-Meteo Hourly Data + DO Feature Engineering

1. Download hourly weather data for Eluru and Nellore → `data/open_meteo_24.csv`
2. Join each DO measurement with the nearest prior hourly weather observation
3. Engineer cumulative lag features per the literature → `data/weather_features.csv`

In [1]:
import requests
import pandas as pd

regions = {
    "Eluru":  {"lat": 16.7119, "lon": 81.0949},
    "Nellore": {"lat": 14.4426, "lon": 79.9865},
}

start_date = "2021-06-15"
end_date   = "2026-02-15"

hourly_vars = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall", "weather_code", "surface_pressure",
    "et0_fao_evapotranspiration", "vapor_pressure_deficit", "wind_speed_10m",
    "wind_direction_10m", "shortwave_radiation", "direct_radiation",
]

dfs = []

for region_name, coords in regions.items():
    print(f"Downloading: {region_name} ({start_date} to {end_date})...")
    params = {
        "latitude":   coords["lat"],
        "longitude":  coords["lon"],
        "start_date": start_date,
        "end_date":   end_date,
        "hourly":     ",".join(hourly_vars),
        "timezone":   "Asia/Kolkata",
    }
    response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)
    if response.status_code == 200:
        df = pd.DataFrame(response.json().get("hourly", {}))
        df["region"] = region_name
        dfs.append(df)
        print(f"  {len(df)} hourly rows")
    else:
        print(f"  Error {response.status_code}: {response.text[:200]}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    df_all["time"] = pd.to_datetime(df_all["time"])
    df_all.to_csv("data/open_meteo_24.csv", index=False)
    print(f"\nSaved {len(df_all)} rows to data/open_meteo_24.csv")
else:
    print("No data collected.")

Downloading: Eluru (2021-06-15 to 2026-02-15)...
  40968 hourly rows
Downloading: Nellore (2021-06-15 to 2026-02-15)...
  40968 hourly rows

Saved 81936 rows to data/open_meteo_24.csv


## Feature Engineering

For every DO measurement, merge with the closest prior hourly weather observation
(per region), then compute cumulative lag features:

| Feature | Window | Source variable |
|---|---|---|
| `shortwave_sum_24h` | prior 24 h | shortwave_radiation |
| `shortwave_sum_48h` | prior 48 h | shortwave_radiation |
| `temp_max_24h` | prior 24 h | temperature_2m |
| `wind_min_overnight` | 20:00 prev day → 06:00 meas day | wind_speed_10m |
| `precip_sum_48h` | prior 48 h | precipitation |
| `vpd_mean_24h` | prior 24 h | vapor_pressure_deficit |

In [2]:
import pandas as pd
import numpy as np

# ── 1. Load hourly weather ────────────────────────────────────────────────────
df_wx = pd.read_csv("data/open_meteo_24.csv")
df_wx["time"] = pd.to_datetime(df_wx["time"])
df_wx = df_wx.sort_values(["region", "time"]).reset_index(drop=True)

# ── 2. Rolling cumulative features per region ─────────────────────────────────
# closed="left" means the window excludes the current hour (strictly prior hours)
feat_parts = []
for region, grp in df_wx.groupby("region"):
    grp = grp.set_index("time").sort_index()

    grp["shortwave_sum_24h"] = grp["shortwave_radiation"].rolling("24h", closed="left").sum()
    grp["shortwave_sum_48h"] = grp["shortwave_radiation"].rolling("48h", closed="left").sum()
    grp["temp_max_24h"]      = grp["temperature_2m"].rolling("24h", closed="left").max()
    grp["precip_sum_48h"]    = grp["precipitation"].rolling("48h", closed="left").sum()
    grp["vpd_mean_24h"]      = grp["vapor_pressure_deficit"].rolling("24h", closed="left").mean()

    grp["region"] = region
    feat_parts.append(grp.reset_index())

df_wx_feat = pd.concat(feat_parts, ignore_index=True)

# ── 3. Overnight min wind speed: 20:00 prev day → 06:00 measurement day ───────
df_wx_night = df_wx_feat[
    (df_wx_feat["time"].dt.hour >= 20) | (df_wx_feat["time"].dt.hour < 6)
].copy()

# Each overnight row belongs to the calendar day whose 06:00 closes the window
df_wx_night["overnight_date"] = df_wx_night["time"].dt.normalize()
df_wx_night.loc[df_wx_night["time"].dt.hour >= 20, "overnight_date"] += pd.Timedelta(days=1)

overnight_wind = (
    df_wx_night
    .groupby(["region", "overnight_date"])["wind_speed_10m"]
    .min()
    .reset_index(name="wind_min_overnight")
)

# ── 4. Load DO measurements ───────────────────────────────────────────────────
df_wq = pd.read_csv("data/water_quality.csv")
df_wq = df_wq[df_wq["DO (mg/L)"].notna()].copy()
df_wq["datetime"] = pd.to_datetime(
    df_wq["Date of data collection"] + " " + df_wq["Time of data collection"],
    format="%m/%d/%Y %H:%M",
)

# ── 5. Merge each DO row with nearest prior hourly weather (per region) ────────
merged_parts = []
for region in df_wq["region"].unique():
    wq_r = df_wq[df_wq["region"] == region].sort_values("datetime")
    wx_r = df_wx_feat[df_wx_feat["region"] == region].sort_values("time")
    m = pd.merge_asof(
        wq_r, wx_r,
        left_on="datetime",
        right_on="time",
        suffixes=("", "_wx"),
        direction="backward",
    )
    merged_parts.append(m)

df_merged = pd.concat(merged_parts, ignore_index=True)

# Drop duplicate region column produced by merge_asof
if "region_wx" in df_merged.columns:
    df_merged.drop(columns=["region_wx"], inplace=True)

# ── 6. Join overnight min wind ────────────────────────────────────────────────
df_merged["meas_date"] = df_merged["datetime"].dt.normalize()
df_merged = df_merged.merge(
    overnight_wind,
    left_on=["region", "meas_date"],
    right_on=["region", "overnight_date"],
    how="left",
)
df_merged.drop(columns=["overnight_date"], inplace=True)

# ── 7. Select columns and save ────────────────────────────────────────────────
keep_cols = [
    "Sr. No", "datetime", "pond_id", "region", "DO (mg/L)", "Type",
    "time",                               # matched weather timestamp (floor hour)
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "vapor_pressure_deficit", "wind_speed_10m", "wind_direction_10m",
    "shortwave_radiation", "direct_radiation",
    "precipitation", "surface_pressure", "et0_fao_evapotranspiration",
    # --- engineered features ---
    "shortwave_sum_24h",   # cumulative shortwave radiation prior 24 h  (W/m² · h)
    "shortwave_sum_48h",   # cumulative shortwave radiation prior 48 h
    "temp_max_24h",        # maximum temperature prior 24 h  (°C)
    "wind_min_overnight",  # min wind speed overnight window (m/s)
    "precip_sum_48h",      # total precipitation prior 48 h  (mm)
    "vpd_mean_24h",        # mean VPD prior 24 h  (kPa)
]

keep_cols = [c for c in keep_cols if c in df_merged.columns]
df_features = df_merged[keep_cols].copy()

out_path = "data/weather_features.csv"
df_features.to_csv(out_path, index=False)
print(f"Saved {len(df_features)} rows × {len(df_features.columns)} cols → {out_path}")
df_features.head()

Saved 11416 rows × 24 cols → data/weather_features.csv


,Sr. No,datetime,pond_id,region,DO (mg/L),Type,time,temperature_2m,relative_humidity_2m,dew_point_2m,...,direct_radiation,precipitation,surface_pressure,et0_fao_evapotranspiration,shortwave_sum_24h,shortwave_sum_48h,temp_max_24h,wind_min_overnight,precip_sum_48h,vpd_mean_24h
0,12,2021-07-12 06:15:00,pond_a79fbbe7,Eluru,7.1,Morning,2021-07-12 06:00:00,25.9,94,24.9,...,2.0,0.1,998.2,0.02,2541.0,7312.0,29.4,4.1,49.7,0.342500
1,13,2021-07-12 06:30:00,pond_c2428871,Eluru,6.2,Morning,2021-07-12 06:00:00,25.9,94,24.9,...,2.0,0.1,998.2,0.02,2541.0,7312.0,29.4,4.1,49.7,0.342500
2,14,2021-07-12 06:45:00,pond_75cb7fc7,Eluru,1.5,Morning,2021-07-12 06:00:00,25.9,94,24.9,...,2.0,0.1,998.2,0.02,2541.0,7312.0,29.4,4.1,49.7,0.342500
3,15,2021-07-12 17:45:00,pond_c2428871,Eluru,6.6,Evening,2021-07-12 17:00:00,26.9,93,25.5,...,17.0,0.7,996.9,0.09,1838.0,4426.0,28.6,4.1,65.0,0.255000
4,16,2021-07-12 18:30:00,pond_75cb7fc7,Eluru,5.8,Evening,2021-07-12 18:00:00,26.4,94,25.3,...,2.0,1.4,997.7,0.03,1835.0,4417.0,28.6,4.1,65.6,0.250417
